In [ ]:
%cd /content
!rm -rf aibo_v8
!git clone -b colab-stable https://github.com/miya390831-a11y/aibo_v8.git
%cd /content/aibo_v8
!ls

In [ ]:
!nvidia-smi

In [ ]:
# セル2.5: バージョン固定で diffusers / peft / scipy の整合を取る（混在対策）
!pip install -q --upgrade --force-reinstall diffusers==0.31.0
!pip install -q transformers==4.46.0 accelerate==0.34.0 peft==0.13.0
!pip install -q huggingface_hub==0.26.0
!pip install -q scipy==1.13.0

# キャッシュクリア（ランタイム強制終了 → Colab が新セッションで再接続）
import os

os.kill(os.getpid(), 9)

In [ ]:
%cd /content/aibo_v8
import importlib.util
import os
import subprocess
import sys

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

def load_module_from_file(name, path):
    path = os.path.abspath(path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

load_module_from_file('01_config', '01_config.py')
subprocess.run([sys.executable, '02_colab_setup.py'], cwd='.', check=False)

cfg_mod = sys.modules['01_config']
setup_mod = load_module_from_file('02_colab_setup', '02_colab_setup.py')
sys_cfg = cfg_mod.SystemConfig()
bootstrap = setup_mod.ColabBootstrap(sys_cfg)
bootstrap.run()

print('✅ Bootstrap 完了')

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    print('✅ HF Token 設定完了')
except Exception as e:
    print(f'❌ HF Token エラー: {e}')

In [ ]:
%cd /content/aibo_v8
import importlib.util
import sys
import os

ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

def load_module_from_file(name, path):
    spec = importlib.util.spec_from_file_location(name, os.path.abspath(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

main_mod = load_module_from_file('07_main', '07_main.py')
main_mod.run()

### Next.js + FastAPI + ngrok（任意 · `07_main` の Gradio の後に実行）

- **Secrets** に **`NGROK_TOKEN`** を登録（[ngrok ダッシュボードの authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)）。
- **順序**: セル6（FastAPI）→ セル7（`npm install`）→ セル8（`build` / `start` / ngrok）。セル8 の最後に表示される **`UI URL`** から Next を開きます（API は同一オリジンまたは FastAPI の URL を環境に合わせて設定）。

In [ ]:
%cd /content/aibo_v8
# セル6: FastAPI（バックグラウンド）
import subprocess
import time

fastapi_proc = subprocess.Popen(
    ["python", "09_fastapi_server.py"],
    cwd="/content/aibo_v8",
)
time.sleep(10)
print("✅ FastAPI 起動")

In [ ]:
# セル7: Next.js セットアップ
%cd /content/aibo_v8/frontend
!npm install --silent
print("✅ Next.js セットアップ完了")

In [ ]:
# セル8: Next.js 本番ビルド + バックグラウンド起動 + ngrok 公開
# 事前: Colab → 鍵アイコン → Secrets に NGROK_TOKEN（https://dashboard.ngrok.com/get-started/your-authtoken）
%cd /content/aibo_v8/frontend
!npm run build

!pip install -q pyngrok
from google.colab import userdata
from pyngrok import ngrok

_token = userdata.get("NGROK_TOKEN")
if not _token or not str(_token).strip():
    raise RuntimeError("Colab Secrets に NGROK_TOKEN を設定してください（ngrok authtoken）")
ngrok.set_auth_token(str(_token).strip())

import subprocess
import time

nextjs_proc = subprocess.Popen(
    ["npm", "start"],
    cwd="/content/aibo_v8/frontend",
)
time.sleep(15)

public_url = ngrok.connect(3000)
print(f"🌐 UI URL: {public_url}")